# Kalshi Scanner: results

Run the cells top to bottom. This only *reads* the data the GitHub Action has been collecting; it places no orders.

**One-time setup:** create a GitHub personal access token (Settings > Developer settings > Fine-grained tokens, read-only access to `Kalshi-Scanner` contents). In Colab, open the key icon (Secrets), add it as `GITHUB_TOKEN`, and enable notebook access.

In [ ]:
from google.colab import userdata
import subprocess, os

token = userdata.get('GITHUB_TOKEN')
if os.path.exists('Kalshi-Scanner'):
    subprocess.run(['git', '-C', 'Kalshi-Scanner', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', f'https://{token}@github.com/UseSoap/Kalshi-Scanner.git'], check=True)
%cd Kalshi-Scanner
!pip install -q -r requirements.txt

## Summary: does the hit rate beat the price?

In [ ]:
!python -m kalshi_scanner report

## Raw paper trades

In [ ]:
import pandas as pd

trades = pd.read_csv('data/trades.csv')
trades['entry_price'] = trades['entry_price'].astype(float)
trades.sort_values('first_seen', ascending=False).head(25)

## Hit rate vs. implied probability by price bucket

If the blue (actual) bars sit above the orange (implied) bars by more than fees, there may be an edge. With small samples, they will bounce around a lot.

In [ ]:
import matplotlib.pyplot as plt

done = trades[trades['status'] == 'settled'].copy()
done['won'] = done['won'].astype(int)
done['bucket'] = pd.cut(done['entry_price'], [95, 96, 97, 98, 99, 100], right=False)
g = done.groupby('bucket', observed=True).agg(n=('won', 'size'), actual=('won', 'mean'), implied=('entry_price', lambda s: s.mean() / 100))
print(g)
g[['actual', 'implied']].plot.bar(figsize=(8, 4), ylim=(0.9, 1.0), rot=0)
plt.ylabel('probability'); plt.show()

## Cumulative paper P&L

In [ ]:
done = done.sort_values('settled_at')
done['pnl_usd'] = done['pnl_usd'].astype(float)
done['cum_pnl'] = done['pnl_usd'].cumsum()
done.plot(x='settled_at', y='cum_pnl', figsize=(9, 4), legend=False)
plt.ylabel('USD (100 contracts per trade, after fees)'); plt.show()